# R&D 특구 유사 과제 검색 프로토타입

준비 중인 연구 주제를 입력하면, R&D 특구 소속 기관이 수행한 유사 과제와 그 과제를 수행한 연구자를
찾아줍니다.

## 1. 데이터 적재 — 특구 소속 기관이 수행한 과제만 추출

In [ ]:
import pandas as pd

DATA_DIR = "../01_data"

companies = pd.read_csv(f"{DATA_DIR}/특구입주기업현황.csv", encoding="cp949")
tasks_raw = pd.read_csv(f"{DATA_DIR}/이알앤디_과제정보.csv", encoding="cp949")

print("특구입주기업현황:", len(companies), "건")
print("이알앤디_과제정보 (원본):", len(tasks_raw), "건")

tasks = tasks_raw[tasks_raw["보안과제여부"] != "Y"].copy()
print("보안과제 제외 후:", len(tasks), "건")
assert len(tasks) == 11783, f"보안과제 제외 후 건수가 예상과 다름: {len(tasks)}"

In [ ]:
companies = companies.rename(columns={"지역": "특구지역"})
companies["_key"] = companies["기관명"].str.strip()
tasks["_key"] = tasks["주관기관명"].str.strip()

companies_dedup = companies.drop_duplicates(subset="_key")

df = (
    tasks.merge(companies_dedup[["_key", "특구지역"]], on="_key", how="inner")
    .drop(columns=["_key"])
    .reset_index(drop=True)
)

print("조인 결과:", len(df), "건 /", df["주관기관명"].nunique(), "개 기관")
assert len(df) == 1680, f"조인 건수가 예상과 다름: {len(df)}"
assert df["주관기관명"].nunique() == 27, f"조인 기관 수가 예상과 다름: {df['주관기관명'].nunique()}"
assert df["연구자번호"].nunique() == 1464, f"연구자 수가 예상과 다름: {df['연구자번호'].nunique()}"

## 2. 과제명 벡터화 — 검색·클러스터링·연구자 추천이 공유하는 파이프라인

In [ ]:
_name_org_ids = (
    df.drop_duplicates(subset=["연구책임자명", "주관기관명", "연구자번호"])
    .groupby(["연구책임자명", "주관기관명"])["연구자번호"]
    .nunique()
)
DUP_NAME_ORG_PAIRS = set(_name_org_ids[_name_org_ids > 1].index)
print("동명이인 표기 대상 (이름+기관) 쌍:", len(DUP_NAME_ORG_PAIRS))
assert len(DUP_NAME_ORG_PAIRS) == 11, f"동명이인 쌍 수가 예상과 다름: {len(DUP_NAME_ORG_PAIRS)}"

def display_researcher_name(row, dup_pairs=DUP_NAME_ORG_PAIRS):
    key = (row["연구책임자명"], row["주관기관명"])
    if key in dup_pairs:
        return f"{row['연구책임자명']}({str(int(row['연구자번호']))[-4:]})"
    return row["연구책임자명"]

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df["과제명"])
print("TF-IDF 행렬 크기:", tfidf_matrix.shape)

svd = TruncatedSVD(n_components=2, random_state=42)
coords = svd.fit_transform(tfidf_matrix)

N_CLUSTERS = 6  # Task 9에서 화면2 산점도를 보고 최종 확정

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
clusters = kmeans.fit_predict(coords)

df["_cluster"] = clusters
print("클러스터별 건수:")
print(df["_cluster"].value_counts().sort_index())

In [ ]:
sizes = df["_cluster"].value_counts()
assert len(sizes) == N_CLUSTERS, f"빈 클러스터 존재: {len(sizes)}/{N_CLUSTERS}"
assert (sizes > 0).all()

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

SIMILARITY_THRESHOLD = 0.15  # Task 9에서 최종 확정
TOP_N_TASKS = 5              # Task 9에서 최종 확정

def search_similar_tasks(query, threshold=SIMILARITY_THRESHOLD, top_n=TOP_N_TASKS):
    if len(df) == 0:
        return {"empty_corpus": True}

    if query is None or not query.strip():
        return {"empty_corpus": False, "blank_query": True}

    query_vec = vectorizer.transform([query])
    sims = cosine_similarity(query_vec, tfidf_matrix).flatten()
    query_coord = svd.transform(query_vec)
    query_cluster = int(kmeans.predict(query_coord)[0])

    mask = sims >= threshold
    matched_idx = np.where(mask)[0]

    if len(matched_idx) == 0:
        fallback_idx = np.argsort(-sims)[:3]
        table = df.iloc[fallback_idx][["과제명", "주관기관명", "선정년도"]].copy()
        table["유사도"] = sims[fallback_idx]
        table["클러스터"] = df.iloc[fallback_idx]["_cluster"].values
        return {
            "empty_corpus": False, "blank_query": False, "reference_only": True,
            "count": 0, "orgs": [], "table": table, "cluster": query_cluster, "sims": sims,
        }

    ordered = matched_idx[np.argsort(-sims[matched_idx])][:top_n]
    table = df.iloc[ordered][["과제명", "주관기관명", "선정년도"]].copy()
    table["유사도"] = sims[ordered]
    table["클러스터"] = df.iloc[ordered]["_cluster"].values
    return {
        "empty_corpus": False, "blank_query": False, "reference_only": False,
        "count": len(matched_idx),
        "orgs": sorted(df.iloc[matched_idx]["주관기관명"].unique().tolist()),
        "table": table, "cluster": query_cluster, "sims": sims,
    }

print("search_similar_tasks 함수 정의 완료")

In [ ]:
print("=== 예시 3문장 테스트 ===")
for q in ["인공지능 기반 이미지 분석 기술 개발", "이차전지 소재 개발 연구", "탄소중립 에너지 저장 시스템 개발"]:
    r = search_similar_tasks(q)
    print(q, "-> count:", r["count"], "reference_only:", r["reference_only"], "cluster:", r["cluster"])
    assert r["count"] > 0 or r["reference_only"], f"'{q}'에 대해 표시할 행이 없음"

print("\n=== 엣지 케이스 테스트 ===")
for q in ["", "   ", "가", "a"]:
    r = search_similar_tasks(q)
    print(repr(q), "-> blank:", r.get("blank_query"), "reference_only:", r.get("reference_only"))

print("\n=== 모든 테스트 통과 ===")

In [ ]:
TOP_N_RESEARCHERS = 5  # Task 9에서 최종 확정
RANK_OPTIONS = ["최고 유사도", "유사 과제 건수", "합산 점수"]

def recommend_researchers(sims, threshold=SIMILARITY_THRESHOLD, rank_by="최고 유사도", top_n=TOP_N_RESEARCHERS):
    work = df.copy()
    work["유사도"] = sims
    work["임계값이상"] = work["유사도"] >= threshold

    counts_above = work.groupby("연구자번호")["임계값이상"].sum()
    idxmax_all = work.groupby("연구자번호")["유사도"].idxmax()
    rep = work.loc[
        idxmax_all,
        ["연구자번호", "연구책임자명", "주관기관명", "특구지역", "과제명", "유사도"],
    ].rename(columns={"과제명": "대표 유사 과제명"}).set_index("연구자번호")
    sum_above = work[work["임계값이상"]].groupby("연구자번호")["유사도"].sum()

    researchers = rep.copy()
    researchers["과제수"] = counts_above.reindex(researchers.index).fillna(0).astype(int)
    researchers["합산점수"] = sum_above.reindex(researchers.index).fillna(0.0)
    researchers = researchers.reset_index()

    qualifying = researchers[researchers["과제수"] > 0]

    if len(qualifying) == 0:
        selected = researchers.sort_values("유사도", ascending=False).head(3)
        reference_only = True
    else:
        if rank_by == "유사 과제 건수":
            selected = qualifying.sort_values(["과제수", "유사도"], ascending=[False, False])
        elif rank_by == "합산 점수":
            selected = qualifying.sort_values("합산점수", ascending=False)
        else:
            selected = qualifying.sort_values("유사도", ascending=False)
        selected = selected.head(top_n)
        reference_only = False

    selected = selected.copy()
    selected["연구책임자명"] = selected.apply(display_researcher_name, axis=1)
    display_cols = ["연구책임자명", "주관기관명", "특구지역", "대표 유사 과제명", "유사도", "과제수"]
    return {"reference_only": reference_only, "table": selected[display_cols]}

print("recommend_researchers 함수 정의 완료")

## 3. 화면 1 — 유사 과제 검색 + 연구자 추천

In [ ]:
print("=== 예시 3문장 × 랭킹 3종 = 9회 호출 확인 ===")
example_queries = ["인공지능 기반 이미지 분석 기술 개발", "이차전지 소재 개발 연구", "탄소중립 에너지 저장 시스템 개발"]

for q in example_queries:
    r_search = search_similar_tasks(q)
    for rank_by in RANK_OPTIONS:
        r_rec = recommend_researchers(r_search["sims"], rank_by=rank_by)
        n_rows = len(r_rec["table"])
        print(q, "|", rank_by, "-> rows:", n_rows, "reference_only:", r_rec["reference_only"])
        assert n_rows >= 1, f"'{q}' / '{rank_by}' 조합에서 연구자 추천 행이 0개"

print("\n=== 모든 9회 조합 테스트 통과 ===")

## 4. 화면 2 — 전체 과제 클러스터 지도

In [ ]:
print("=== 동명이인 표기 확인 (Task 2) ===")
researchers = df.drop_duplicates(subset=["연구자번호"])[["연구자번호", "연구책임자명", "주관기관명"]]
labeled = [r for _, r in researchers.iterrows() if "(" in display_researcher_name(r)]
print("괄호 표기가 붙은 연구자 수 (전체 1,464명 중):", len(labeled))
assert len(labeled) > 0, "동명이인 표기 로직이 한 번도 발생하지 않음 — 로직 점검 필요"
print("=== 동명이인 표기 확인 통과 ===")

## 5. 알려진 한계

TF-IDF 표면 단어 매칭 방식이라, 같은 의미라도 다른 표현("인공지능" vs "AI")을 쓰면 유사 과제로 잡히지 않을 수 있습니다. 반대로 짧은 과제명에서 흔한 단어 하나만 겹쳐도 유사도가 실제보다 높게 나올 수 있습니다.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

text_input = widgets.Text(description="과제 주제:", placeholder="준비 중인 과제의 주제를 입력하세요", layout=widgets.Layout(width="500px"))
rank_dropdown = widgets.Dropdown(options=RANK_OPTIONS, value="최고 유사도", description="랭킹 기준:")
search_button = widgets.Button(description="검색", button_style="primary")
output_area = widgets.Output()

def render_results(query, rank_by):
    with output_area:
        clear_output()
        result = search_similar_tasks(query)

        if result.get("empty_corpus"):
            print("특구 소속 기관이 수행한 과제 데이터가 없습니다.")
            return
        if result.get("blank_query"):
            print("검색어를 입력해주세요.")
            return

        print(f"내 주제가 속한 것으로 추정되는 클러스터 번호: {result['cluster']}")

        if result["reference_only"]:
            print("정확히 일치하는 유사 과제를 찾지 못했습니다. 아래는 유사도가 가장 높았던 3건입니다(참고용).")
        else:
            print(f"유사 과제 건수: {result['count']}건")
            print("수행 기관:", ", ".join(result["orgs"]))
        display(result["table"])

        rec = recommend_researchers(result["sims"], rank_by=rank_by)
        if rec["reference_only"]:
            print("참고용 — 임계값 미달: 유사도가 가장 높은 연구자 3명입니다.")
        print("연구자 추천")
        display(rec["table"])

def on_search_clicked(_):
    render_results(text_input.value, rank_dropdown.value)

def on_rank_changed(_):
    render_results(text_input.value, rank_dropdown.value)

search_button.on_click(on_search_clicked)
rank_dropdown.observe(on_rank_changed, names="value")

display(widgets.HBox([text_input, rank_dropdown, search_button]))
display(output_area)

In [ ]:
for q in ["인공지능 기반 이미지 분석 기술 개발", "이차전지 소재 개발 연구", "탄소중립 에너지 저장 시스템 개발", "", "가"]:
    text_input.value = q
    for rank_by in RANK_OPTIONS:
        rank_dropdown.value = rank_by
        on_search_clicked(None)  # output_area에 렌더링, 예외 발생 시 여기서 즉시 실패
print("모든 조합 실행 완료 — 예외 없음")

In [ ]:
import plotly.graph_objects as go

def render_cluster_map(query):
    fig = go.Figure()
    for c in sorted(df["_cluster"].unique()):
        sub = df[df["_cluster"] == c]
        fig.add_trace(go.Scatter(
            x=coords[sub.index, 0], y=coords[sub.index, 1],
            mode="markers", name=f"클러스터 {c}",
            marker=dict(size=6),
            hovertext=[
                f"{row['과제명']}<br>{row['주관기관명']}<br>{row['특구지역']}<br>클러스터 {c}"
                for _, row in sub.iterrows()
            ],
            hoverinfo="text",
        ))

    result = search_similar_tasks(query)
    if not result.get("blank_query") and not result.get("empty_corpus"):
        query_vec = vectorizer.transform([query])
        qx, qy = svd.transform(query_vec)[0]
        fig.add_trace(go.Scatter(
            x=[qx], y=[qy], mode="markers+text",
            marker=dict(size=16, symbol="star", color="black"),
            text=[f"내 주제 (클러스터 {result['cluster']})"], textposition="top center",
            name="내 주제",
        ))
        assert result["cluster"] in df["_cluster"].unique(), "화면1 클러스터 번호가 화면2 클러스터 목록에 없음"

    fig.update_layout(title="전체 과제 클러스터 지도", xaxis_title="SVD 1", yaxis_title="SVD 2")
    fig.show()
    return fig

render_cluster_map("이차전지 소재 개발 연구")

In [ ]:
for q in ["인공지능 기반 이미지 분석 기술 개발", "이차전지 소재 개발 연구", "탄소중립 에너지 저장 시스템 개발"]:
    r = search_similar_tasks(q)
    row_clusters = set(r["table"]["클러스터"]) if not r["reference_only"] else set(r["table"]["클러스터"])
    all_clusters = set(df["_cluster"].unique())
    assert row_clusters.issubset(all_clusters), f"'{q}' 결과의 클러스터 번호가 화면2 목록에 없음: {row_clusters - all_clusters}"
print("화면1/화면2 클러스터 번호 대조 완료 — 불일치 없음")

## 6. Task 9 — 파라미터 최종 확정 (임계값 / k / Top-N)

발표에 사용할 예시 주제 3개를 최종 확정한다(Task 1~8부터 계속 사용해 온 문장과 동일하며, 이미 매칭 건수가
확인된 문장이므로 그대로 유지한다):

- `"인공지능 기반 이미지 분석 기술 개발"`
- `"이차전지 소재 개발 연구"`
- `"탄소중립 에너지 저장 시스템 개발"`

아래 셀들에서 이 3문장을 기준으로 `SIMILARITY_THRESHOLD`, `N_CLUSTERS`, `TOP_N_TASKS`,
`TOP_N_RESEARCHERS` 값을 실측 데이터로 재검증하고 최종 확정한다. 판단 근거는 각 셀의 출력에 함께
남긴다.

In [ ]:
for t in [0.10, 0.15, 0.20, 0.25]:
    print(f"--- threshold={t} ---")
    for q in ["인공지능 기반 이미지 분석 기술 개발", "이차전지 소재 개발 연구", "탄소중립 에너지 저장 시스템 개발"]:
        r = search_similar_tasks(q, threshold=t)
        print(" ", q, "-> count:", r["count"], "reference_only:", r["reference_only"])

print()
print("=== SIMILARITY_THRESHOLD 최종 판단 ===")
print("0.20에서도 세 문장 모두 count>0(12/7/1)이라, Step 2 문구('count>0을 만족하는 가장 엄격한")
print("임계값')만 기계적으로 적용하면 0.20을 골라야 한다. 하지만 '탄소중립 에너지 저장 시스템 개발'은")
print("count=1로 여유가 거의 없어, 발표 환경의 라이브러리 버전이나 데이터 미세 변경만으로도 0건(참고용")
print("폴백)으로 뒤집힐 위험이 있다. 또한 Step 4의 목표(표가 항상 꽉 차 보이도록)를 함께 고려하면,")
print("0.15에서는 세 문장의 매칭 건수가 30/8/7건으로 모두 TOP_N_TASKS=5 이상이라 세 경우 모두 표가")
print("5행으로 꽉 차는 반면, 0.20에서는 세 번째 문장이 1건뿐이라 5행 중 1행만 채워져 발표 화면이")
print("빈약해 보인다.")
print("=> SIMILARITY_THRESHOLD = 0.15 로 유지한다 (변경하지 않음).")

In [ ]:
print("=== N_CLUSTERS 시각 점검 (Step 3) ===")
sizes = df["_cluster"].value_counts().sort_index()
print(sizes)
print("최대 클러스터 비율:", round(sizes.max() / len(df) * 100, 1), "%")
print("최소 클러스터 비율:", round(sizes.min() / len(df) * 100, 1), "%")

render_cluster_map("이차전지 소재 개발 연구")

print()
print("=== N_CLUSTERS 최종 판단 ===")
print("6개 클러스터의 건수는 84~584건(전체 1,680건 중 5.0%~34.8%)으로 분포한다.")
print("하나의 클러스터가 90% 이상을 차지할 만큼 몰려 있지 않고, 반대로 1건짜리 클러스터로 과도하게")
print("쪼개진 것도 아니다. 산점도 상에서도 6개 색상이 고르게 섞여 나타난다.")
print("=> N_CLUSTERS = 6 으로 유지한다 (변경하지 않음).")

In [ ]:
print("=== TOP_N_TASKS / TOP_N_RESEARCHERS 최종 판단 ===")
print("3절(cell-12)에서 이미 확인했듯, 예시 3문장 x 랭킹 3종 = 9회 호출 모두 rows=5로 연구자 추천")
print("표가 항상 꽉 찬다. threshold=0.15 기준 과제 매칭 건수도 30/8/7건으로 TOP_N_TASKS=5를 채우기에")
print("충분하다. 발표 화면에서 스크롤이 생기지 않으면서도 표가 항상 꽉 차 보이는 5 전후 값이 이미")
print("만족되므로 조정할 이유가 없다.")
print("=> TOP_N_TASKS = 5, TOP_N_RESEARCHERS = 5 로 유지한다 (변경하지 않음).")

### Task 9 결론

| 파라미터 | 최종값 | 근거 |
|---|---|---|
| `SIMILARITY_THRESHOLD` | 0.15 | 위 스윕 결과, 세 예시 문장 모두 count>0이며 여유 있게 매칭(30/8/7건). 0.20도 기계적으로는 "count>0" 조건을 만족하지만 한 문장이 count=1로 여유가 없고, 표가 채워지지 않아(1/5행) 채택하지 않음. |
| `N_CLUSTERS` | 6 | 클러스터 크기가 84~584건(전체의 5.0%~34.8%)으로, 한쪽으로 90% 이상 몰리지도 않고 과도하게 쪼개지지도 않음. |
| `TOP_N_TASKS` | 5 | threshold=0.15에서 예시 3문장 매칭 건수(30/8/7)가 모두 5 이상이라 표가 항상 꽉 찬다. |
| `TOP_N_RESEARCHERS` | 5 | 3절(cell-12)의 9회 조합 테스트에서 이미 매번 rows=5로 확인됨. |

이 값들은 발표 당일까지 변경하지 않는다.